# Week 9 Lab: Look Inside an LLM
**COMP1023 Natural Language Technologies** | about 45 minutes | free Colab, no GPU needed

Four short experiments. The first three use one small model (GPT-2); the last runs Llama with Ollama. Run each cell from top to bottom, then answer the questions in the text boxes.

1. **Memory:** how much memory does a model need?
2. **Tokens:** how does the model see your text?
3. **Temperature:** how does the model choose the next word?
4. **Ollama:** run a real assistant model and check its memory

## Setup

In [1]:
!pip -q install transformers psutil


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: c:\Users\eamon\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [2]:
import psutil, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
print("Ready. This computer has", round(psutil.virtual_memory().total / 1e9, 1), "GB of memory.")

: 

## 1. Memory

Rule from the lecture: **memory = number of parameters x bytes per parameter**

| Precision | Bytes per parameter |
|---|---|
| 32-bit | 4 |
| 16-bit | 2 |
| 8-bit | 1 |
| 4-bit | 0.5 |

Example: a 7 billion parameter model in 16-bit needs 7 x 2 = **14 GB**.

In [1]:
def memory_gb(billions_of_parameters, precision_bits):
    # Map the numerical bit precision to its corresponding byte multiplier
    precision_map = {
        32: 4.0,
        16: 2.0,
        8: 1.0,
        4: 0.5
    }

    try:
        bits = int(precision_bits)
    except ValueError:
        raise ValueError("Please enter a valid integer for precision (e.g., 32, 16, 8, or 4).")

    if bits not in precision_map:
        raise ValueError("Precision must be one of: 32, 16, 8, or 4.")

    return billions_of_parameters * precision_map[bits]

# Ask the user for input dynamically
try:
    parameters_input = float(input("Enter the number of parameters (in billions, e.g., 70): "))
    precision_input = int(input("Enter precision in bits (32, 16, 8, 4): "))

    result = memory_gb(parameters_input, precision_input)
    print(f"\nEstimated memory requirement: {result} GB")
except ValueError as e:
    print(f"\nError: {e}")


Error: could not convert string to float: ''


**Check with a real model.** GPT-2 is small, so we can measure its memory and compare it with our rule.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print("Parameters:", f"{n_params:,}")

# Dynamically predict using our updated integer-based bits precision (32 instead of 4)
print("Predicted memory (32-bit):", round(memory_gb(n_params / 1e9, 32), 3), "GB")
print("Measured memory  (32-bit):", round(model.get_memory_footprint() / 1e9, 3), "GB")

model = model.half()      # store each number in 16 bits instead of 32
print("Measured memory  (16-bit):", round(model.get_memory_footprint() / 1e9, 3), "GB")
model = model.float()     # back to 32-bit for the next parts

**Question 1.** (a) Is the prediction close to the measurement? (b) What happened to the memory when we used 16 bits, and why? (c) Using the rule, would a 70 billion parameter model in 4-bit fit on this computer?

*Your answer:*

## 2. Tokens

The model does not read words. It reads **tokens**, which are pieces of words.

In [ ]:
def show_tokens(text):
    ids = tok.encode(text)
    print(len(ids), "tokens for", len(text.split()), "words:")
    print([tok.decode([i]) for i in ids])

show_tokens("The girl kicked the ball.")
show_tokens("hallucination")
show_tokens("recieve")
show_tokens("Tá an aimsir go maith inniu.")   # Irish: The weather is good today.

**Try it.** Write one sentence in English and the same sentence in another language you know.

In [ ]:
show_tokens("Hello dear students, do you like this module?")
show_tokens("Ciao cari studenti, vi piace questo modulo?")

**Question 2.** Which text needed more tokens per word, and why does this matter for cost and for the context window?

*Your answer:*


## 3. Temperature

The model gives every possible next token a probability. **Temperature** changes how the choice is made: low means safe and repeatable, high means surprising.

In [ ]:
def write(prompt, temperature):
    ids = tok(prompt, return_tensors="pt")
    out = model.generate(**ids, max_new_tokens=15, do_sample=True,
                         temperature=temperature, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0], skip_special_tokens=True)

for t in [0.2, 1.0, 2.0]:
    print("Temperature", t)
    for _ in range(2):
        print("  ", write("The boy kicked the", t))

**Question 3.** What changes as the temperature goes up? Which temperature would you use for a factual answer, and which for a poem? (GPT-2 is a small model from 2019, so even normal text can be a little odd.)

*Your answer:*


## 4. Run a real assistant with Ollama

So far we used GPT-2, a small model from 2019. Now we run **Llama 3.2 (1 billion parameters)**, an assistant model of the kind we saw in the lecture. The steps below come from the Ollama lab: two terminals, one runs the server and one runs the model.

First we install a programme that lets us use terminals inside Colab.

In [ ]:
!pip install colab-xterm #https://pypi.org/project/colab-xterm/
%load_ext colabxterm

Now run the first terminal. When it opens, type the two lines below **one at a time** (leave out the `#`). Be patient: you are running an LLM on a free service.

In [ ]:
# Install zstd dependency, then install Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
%xterm
 # curl https://ollama.ai/install.sh | sh
 # ollama serve

Now run a second terminal. In this one we start a client, which downloads and runs Llama.

In [ ]:
%xterm

# ollama run llama3.2:1b

Chat with the model in the terminal for a minute, then come back here. Below is code to talk to the same model from Python. We will use this next week.

In [ ]:
!ollama list

**Memory check.** Look at the SIZE column above. Llama 3.2 1B has about 1.24 billion parameters. Use the rule from Part 1:

In [ ]:
%pip install -U langchain-ollama

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

llm = OllamaLLM(model="llama3.2:1b")

chain = prompt | llm

chain.invoke({"question": "How can we get to the moon?"})

**Question 4.** (a) Which precision (16, 8 or 4-bit) is closest to the SIZE shown by `ollama list`? (b) Ask the model the same question twice. Are the answers identical? Link this to what you saw about temperature in Part 3.

*Your answer:*


## Finished
Hand in this notebook with your four answers.